# Point-by-point refined map visualisation notebook  
__Written by Haixing Fang, Jon Wright and James Ball__  
__Date: 21/02/2025__

Convert the multivalued voxel refinement to inspectable TensorMaps, using the same conditional measured-group split checks as the selected reconstruction. Full and split estimates require >6 measured unique events, finite positive-volume geometry and finite non-placeholder strain. The split orientation screen remains 0.5 degrees. This is conditional repeatability, not blind validation or certified absolute strain. The recommended orientation-only map additionally requires tomography support.

In [ ]:
# this cell is tagged with parameters
from esrf_pathlib import ESRFPath as Path
import os, sys, importlib.metadata
# As in S3DXRD: optional checkout folder within CHECKOUT_PATH.
IMAGED11_PATH = '/home/esrf/charles27/Code/ImageD11'  # checkout folder within CHECKOUT_PATH
ANRI_PATH = '/home/esrf/charles27/Code/anri'
CHECKOUT_PATH = None
phase_str = 'Fe'
par_file = None  # None: use the DataSet calibration
anri_output_path = None  # None: <ds.analysispath>/anri
cif_file = '/data/id11/archive/file_cif/Fe.cif'
THREADS = 48
SEED_FRACTIONS = (0.993, 0.999, 0.9999)
ACTIVE_FRACTION = 0.999
LOCAL_POPULATION_MIN = 0.1

# Saved DataSet from the mapping/refinement run, including its run folder.
dset_path = '/data/visitor/ma7056/id11/20260624/PROCESSED_DATA/pureIron_2_heating_3/pureIron_2_heating_3_sff_after_20min0/anri_imaged11_s3dxrd_4/pureIron_2_heating_3_sff_after_20min0_dataset.h5'
analysispath = None  # retain ds.analysispath and all saved output paths

min_unique = 6


In [ ]:
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'): os.environ[key]='1'
os.environ['NUMBA_NUM_THREADS']=str(THREADS)
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE','false')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION','0.30')
if hasattr(os, 'sched_getaffinity'):
    os.sched_setaffinity(0, set(sorted(os.sched_getaffinity(0))[:THREADS]))
if ANRI_PATH is not None:
    sys.path.insert(0, str(ANRI_PATH))
if IMAGED11_PATH is not None:
    exec(open('/data/id11/nanoscope/install_ImageD11_from_git.py').read())
    PYTHONPATH = setup_ImageD11_from_git(CHECKOUT_PATH, IMAGED11_PATH)
else:
    import site
    from pathlib import Path as LocalPath
    PYTHONPATH = site.getsitepackages()[0]
    for candidate in [LocalPath.cwd(), LocalPath.cwd() / 'ImageD11', *LocalPath.cwd().parents]:
        if (candidate / 'ImageD11' / 'nbGui' / 'anri' / '4_visualise.ipynb').is_file():
            loaded = sys.modules.get('ImageD11')
            if loaded is not None and (getattr(loaded, '__file__', None) is None or LocalPath(loaded.__file__).resolve().parent != candidate / 'ImageD11'):
                raise RuntimeError('Restart this notebook kernel to use the modified ImageD11 checkout')
            sys.path.insert(0, str(candidate))
            PYTHONPATH = str(candidate)
            break
import ImageD11
import h5py, numpy as np
import ImageD11.sinograms.dataset as dataset
import numba
numba.set_num_threads(THREADS)
def log(message):
    print(message,flush=True)
def snapshot(fig,name):
    from IPython.display import display
    import matplotlib.pyplot as plt
    fig.savefig(RUN/(name+'.png'),dpi=120,bbox_inches='tight'); display(fig); plt.close(fig)

os.environ['JAX_PLATFORMS']='cpu'
import os,sys,time,hashlib,subprocess,importlib.metadata
from ImageD11.sinograms.pbp_refine import PBPRefine
from ImageD11.sinograms.point_by_point import PBPMap
from ImageD11.sinograms.tensor_map import TensorMap
from ImageD11 import columnfile,sym_u
import numpy as np,h5py,numba
numba.set_num_threads(4)
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
import anri.utils
anri.utils.setup(n_cpu=1)
import anri.crystal.orientation
OPS=np.asarray(sym_u.cubic().group)
PATCH=subprocess.run(['git','diff','HEAD','--','ImageD11'],cwd=PYTHONPATH,text=True,capture_output=True,check=True).stdout
PATCH_SHA=hashlib.sha256(PATCH.encode()).hexdigest()
def runtime_source_patch(patch):
    # Omit whole notebook diff sections, including saved outputs, not individual lines.
    import re
    lines=[]; include=True
    for line in patch.splitlines(keepends=True):
        if line.startswith('diff --git '):
            include=re.match(r'diff --git (?:a/.*\.ipynb|"a/.*\.ipynb") (?:b/.*\.ipynb|"b/.*\.ipynb")\n?$',line) is None
        if include: lines.append(line)
    return ''.join(lines)
SOURCE_COMMIT=subprocess.run(['git','rev-parse','HEAD'],cwd=PYTHONPATH,text=True,capture_output=True,check=True).stdout.strip()
RUNTIME_PATCH=runtime_source_patch(PATCH)
SOURCE_SHA=hashlib.sha256((SOURCE_COMMIT+'\n'+RUNTIME_PATCH).encode()).hexdigest()
print('Source patch:',PATCH_SHA)


# Load data
## Dataset

In [ ]:
ds = dataset.load(dset_path)
SOURCE_DATASET = Path(dset_path).absolute()
SOURCE_SPARSE = Path(ds.sparsefile)
SOURCE_GRAINS = Path(ds.grainsfile)
PARFILE = Path(par_file if par_file is not None else ds.parfile)
if par_file is not None:
    ds.parfile = str(PARFILE)
NAME, SAMPLE, SCAN = ds.dsname, ds.sample, ds.dset
PHASE = phase_str
DATA = SOURCE_DATASET.parent
RUN = Path(analysispath) if analysispath is not None else Path(ds.analysispath)
ANRI = Path(anri_output_path) if anri_output_path is not None else RUN/'anri'
VALIDATION = RUN/'validation'
VALIDATION.mkdir(parents=True, exist_ok=True)
VARIANT_DATASET = Path(ds.dsfile) if analysispath is None else RUN/(NAME+'_dataset.h5')
ANRI_FIT = ANRI/(NAME+'_fit.h5')
ANRI_POP = ANRI/(NAME+'_populations.h5')
CIF = Path(cif_file)
print(ds)

BRANCHES={'conservative':RUN}


In [ ]:
print(ds)

## Phases
If the parameter file was a json, we can access the unit cells via `ds.phases.unitcells`

In [ ]:
ds.phases = ds.get_phases_from_disk()
ds.phases.unitcells

In [ ]:
ref_ucell = ds.phases.unitcells[phase_str]
from pprint import pprint
pprint(ref_ucell)

# Point-by-point refined map

In [ ]:
def log(message):
    text=time.strftime('%Y-%m-%dT%H:%M:%S%z ')+message
    print(text,flush=True)
    with (VALIDATION/'progress.log').open('a') as f: f.write(text+'\n')
def snap(fig,name):
    fig.savefig(VALIDATION/f'{name}.png',dpi=120,bbox_inches='tight'); display(fig); plt.close(fig)
def read_map(path):
    return columnfile.colfile_from_hdf(str(path),obj=PBPMap(new=True),name='refinedmap')
def validation_inputs(manager,fold):
    import json
    par=Path(manager.dset.parfile)
    paths=[Path(manager.own_filename),Path(manager.icolf_filename),Path(manager.pbpmap_filename),Path(manager.dset.dsfile),par]
    if par.suffix == '.json':
        schema=json.loads(par.read_text())
        entries=[schema.get('geometry',{})]+list(schema.get('phases',{}).values())
        paths += [par.parent/entry['file'] for entry in entries if 'file' in entry]
    digest=hashlib.sha256((repr(fold)+np.__version__+numba.__version__).encode())
    manifest=dict(format='file-bytes-v1',fold=fold,numpy=np.__version__,numba=numba.__version__,files=[])
    for path in paths:
        digest.update(str(path.resolve()).encode())
        file_digest=hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda:stream.read(1<<20),b''):
                digest.update(block); file_digest.update(block)
        stat=path.stat()
        manifest['files'].append(dict(path=str(path.resolve()),sha256=file_digest.hexdigest(),size_bytes=stat.st_size,mtime_ns=stat.st_mtime_ns))
    return digest.hexdigest(),manifest
def recorded_source_patch(directory=None):
    diagnostics=(VALIDATION if directory is None else directory)/'physical_validation_diagnostics.h5'
    assert diagnostics.exists(),'Missing diagnostics provenance; cannot verify cached source'
    with h5py.File(diagnostics,'r') as h:
        assert 'provenance/executed_source_diff' in h,'Missing executed source diff; cannot verify cached source'
        assert h['provenance'].attrs.get('source_commit')==SOURCE_COMMIT,'Source commit changed or missing; use a new validation directory'
        patch=h['provenance/executed_source_diff'][()].decode()
    assert runtime_source_patch(patch)==RUNTIME_PATCH,'Runtime source changed; use a new validation directory'
    return patch
def check_cached_variant(output,input_key,manifest):
    import json
    with h5py.File(output,'r') as h:
        if h.attrs.get('validation_inputs_sha256')!=input_key:
            recorded=h.attrs.get('validation_inputs_manifest_json','Unavailable (legacy cache): cannot identify the changed input or prove semantic equality.')
            raise AssertionError('Inputs changed or provenance missing for %s. Cached key: %s; current key: %s.\nRecorded manifest: %s\nCurrent manifest: %s\nRefusing cached results. Rerun the RESULTS cell to select an input/source-specific directory; do not edit cache hashes.' % (output,h.attrs.get('validation_inputs_sha256'),input_key,recorded,json.dumps(manifest,sort_keys=True)))
        if 'runtime_source_sha256' in h.attrs:
            assert h.attrs['runtime_source_sha256']==SOURCE_SHA,'Runtime source changed (including source commit); rerun the RESULTS cell to select an input/source-specific directory'
        else:
            patch=recorded_source_patch(output.parent)
            assert hashlib.sha256(patch.encode()).hexdigest()==h.attrs.get('executed_source_sha256'),'Legacy cache source hash does not match recorded full patch; cannot verify cached source'
def validation_directory(manager):
    root=RUN/'validation'
    input_key,manifest=validation_inputs(manager,None)
    # Keep exact, source-proven legacy runs usable; never mix changed inputs with their diagnostics.
    if (root/'conservative_seamfixed_refinedmap.h5').exists():
        try:
            check_cached_variant(root/'conservative_seamfixed_refinedmap.h5',input_key,manifest)
            for fold in (0,1):
                output=root/('conservative_split'+str(fold)+'_refinedmap.h5')
                if output.exists():
                    split_key,split_manifest=validation_inputs(manager,fold)
                    check_cached_variant(output,split_key,split_manifest)
            if (root/'physical_validation_diagnostics.h5').exists(): recorded_source_patch(root)
        except AssertionError as error:
            print('Preserving previous validation run:',root,'\n',error)
        else:
            return root
    directory=root/hashlib.sha256((SOURCE_SHA+'\n'+input_key).encode()).hexdigest()
    directory.mkdir(parents=True,exist_ok=True)
    print('Validation outputs:',directory,'(unproven legacy inputs require one fresh refinement)')
    return directory
def run_variant(manager,output,fold=None):
    import json
    input_key,manifest=validation_inputs(manager,fold)
    if output.exists():
        check_cached_variant(output,input_key,manifest)
        return read_map(output)
    if fold is not None:
        group=np.asarray(manager.icolf.spot3d_id)
        assert np.all(np.isfinite(group)) and np.all(group==np.rint(group))
        keep=group.astype(np.int64)%2==fold
        print('Split',fold,'measured 2D peaks:',int(keep.sum()),'of',len(keep),'4D groups:',len(np.unique(group[keep])))
        manager.icolf=manager.icolf.copyrows(keep)
    start=time.monotonic()
    result=manager.run_refine(output_filename=str(output),nthreads=4,nchunks=4,use_cluster=False)
    with h5py.File(output,'r+') as h:
        h.attrs['executed_source_sha256']=PATCH_SHA
        h.attrs['runtime_source_sha256']=SOURCE_SHA
        h.attrs['validation_inputs_sha256']=input_key
        h.attrs['validation_inputs_manifest_json']=json.dumps(manifest,sort_keys=True)
        h.attrs['source_dataset']=manager.dset.dsfile
        h.attrs['split_rule']='all' if fold is None else 'spot3d_id modulo 2 == '+str(fold)
        h.attrs['measured_2d_peaks']=manager.icolf.nrows
        h.attrs['elapsed_seconds']=time.monotonic()-start
    log('Completed '+output.name+' in '+str(round(time.monotonic()-start,2))+' seconds')
    return result
def preferred(raw,initial,mask):
    nv=mask.size
    assert raw.nrows>=nv
    np.testing.assert_array_equal(raw.i[:nv],initial.i[:nv]); np.testing.assert_array_equal(raw.j[:nv],initial.j[:nv])
    raw.choose_best(min_unique)
    rows=np.asarray(raw.best_rows).ravel().copy()
    ubi=raw.ubi.transpose(2,0,1)[:nv]
    good=(initial.ntotal[:nv]>0)&(raw.nuniq[:nv]>min_unique)&np.isfinite(ubi).all((-2,-1))&(np.linalg.det(ubi)>0)
    rows[good]=np.arange(nv)[good]
    chosen=PBPMap(new=True); chosen.nrows=nv
    valid=rows>=0; selected=np.clip(rows,0,raw.nrows-1)
    for key in raw.titles:
        if key in ('i','j'): values=getattr(initial,key)[:nv].copy()
        else:
            values=getattr(raw,key)[selected].copy()
            values[~valid]=0 if key in ('ntotal','nuniq') else np.nan
        chosen.addcolumn(values,key)
    chosen.choose_best(min_unique)
    return chosen
def arrays(pmap,mask,step,uc):
    pmap.choose_best(min_unique)
    t=TensorMap.from_pbpmap(pmap,steps=[1,step,step],phases={0:uc})
    u=TensorMap.map_order_to_recon_order(t.U,0)
    ubi=pmap.best_ubi
    eps=pmap.best_eps
    valid=mask&(pmap.best_nuniq>min_unique)&np.isfinite(ubi).all((-2,-1))&(np.linalg.det(ubi)>0)&np.isfinite(u).all((-2,-1))
    strain_ok=np.isfinite(eps).all((-2,-1))&np.any(eps!=0,axis=(-2,-1))
    return dict(tmap=t,U=u,UBI=ubi,eps=eps,nuniq=pmap.best_nuniq,ntotal=pmap.best_npks,valid=valid,strain_ok=strain_ok)


In [ ]:
RESULTS={}
for label,folder in BRANCHES.items():
    manager_file=Path(os.path.splitext(ds.refmanfile)[0]+'_'+phase_str+'.h5')
    print(str(manager_file))
    r=PBPRefine.from_h5(str(manager_file))
    if label=='conservative': VALIDATION=validation_directory(r)
    r.refinedmap.plot_nuniq_hist()
    r.refinedmap.choose_best(min_unique)
    r.refinedmap.plot_best(min_unique)
    with h5py.File(r.dset.grainsfile,'r') as h: mask=h['masks/'+phase_str][()].astype(bool)
    assert mask.ndim==2 and mask.shape[0]==mask.shape[1]
    uc=r.dset.get_phases_from_disk().unitcells[phase_str]
    if label=='conservative':
        raw=run_variant(r,VALIDATION/'conservative_seamfixed_refinedmap.h5')
    else:
        raw=r.refinedmap
    full=arrays(preferred(raw,r.pbpmap,mask),mask,r.ystep,uc)
    full['mask']=mask; full['step']=r.ystep; full['uc']=uc
    full['spatial_sampling'] = dict(dty_step=r.dset.ystep, omega_bin_width_deg=getattr(r.dset, 'ostep', None), assumed_beam_size=r.beam_size)
    print(label,'supported full/preferred:',int(full['valid'].sum()),'finite non-placeholder strain:',int(np.count_nonzero(full['valid']&full['strain_ok'])))
    splits=[]
    for fold in (0,1):
        split=PBPRefine.from_h5(str(manager_file),load_output=False)
        output=VALIDATION/(label+'_split'+str(fold)+'_refinedmap.h5')
        fitted=run_variant(split,output,fold=fold)
        splits.append(arrays(preferred(fitted,split.pbpmap,mask),mask,split.ystep,uc))
    full['splits']=splits
    RESULTS[label]=full
    del r,split,raw
print('Native measured-group split refinements complete')

# Make TensorMap

Build the single-representative, split-qualified estimate from measured support. Retain the raw multivalued alternatives and completion-only hypotheses separately.

In [ ]:
DIAG=VALIDATION/'physical_validation_diagnostics.h5'
if DIAG.exists():
    recorded_source_patch()
SUMMARIES={}
from pprint import pprint
with h5py.File(DIAG,'a') as h:
    provenance=h.require_group('provenance')
    provenance.attrs.setdefault('source_repository',PYTHONPATH)
    provenance.attrs.setdefault('source_commit',SOURCE_COMMIT)
    provenance.attrs['conditional_not_blind_validation']=True
    provenance.attrs['no_reference_seed_or_spatial_smoothing']=True
    if 'executed_source_diff' not in provenance: provenance.create_dataset('executed_source_diff',data=PATCH,dtype=h5py.string_dtype('utf-8'))
    for label,result in RESULTS.items():
        a,b=result['splits']
        common=result['valid']&a['valid']&b['valid']
        angular=np.full(common.shape,np.nan)
        angular[common]=anri.crystal.orientation.disorientation(a['U'][common],b['U'][common],OPS)
        strain_common=common&result['strain_ok']&a['strain_ok']&b['strain_ok']
        strain_delta=np.full(common.shape,np.nan)
        strain_delta[strain_common]=np.linalg.norm(a['eps'][strain_common]-b['eps'][strain_common],axis=(-2,-1))
        qualified=strain_common&(angular<=0.5)
        volume=np.linalg.det(result['UBI'])
        summary=dict(full_supported=int(result['valid'].sum()),split_supported=int(common.sum()),qualified=int(qualified.sum()),split_angle_p50_p95_p99=np.percentile(angular[common],[50,95,99]) if common.any() else np.full(3,np.nan),split_strain_difference_p50_p95_p99=np.percentile(strain_delta[strain_common],[50,95,99]) if strain_common.any() else np.full(3,np.nan),volume_p01_p50_p99=np.percentile(volume[result['valid']],[1,50,99]) if result['valid'].any() else np.full(3,np.nan))
        print(label, 'summary (angles: deg; strain differences: dimensionless; cell volumes: Angstrom^3)')
        pprint(summary, width=100)
        g=h.require_group(label)
        for key,value in summary.items(): g.attrs[key]=value
        for key,value in [('manual_support',result['mask']),('full_support',result['valid']),('qualified_support',qualified),('split_angle_deg',angular),('split_strain_difference',strain_delta),('full_nuniq',result['nuniq']),('full_ntotal',result['ntotal']),('split0_nuniq',a['nuniq']),('split1_nuniq',b['nuniq']),('full_cell_volume',volume)]:
            if key in g: np.testing.assert_allclose(g[key][()],value,rtol=1e-8,atol=1e-12,equal_nan=True)
            else: g.create_dataset(key,data=value,compression='lzf')
        # Keep only primary standard maps so derived orientation fields are recomputed after masking.
        maps={key:np.array(result['tmap'][key],copy=True) for key in ('UBI','nuniq')}
        eps=np.array(TensorMap.recon_order_to_map_order(result['eps']),copy=True)
        invalid=~TensorMap.recon_order_to_map_order(qualified)
        maps['UBI'][invalid]=np.nan; eps[invalid]=np.nan
        maps['eps_sample']=eps
        maps['phase_ids']=TensorMap.recon_order_to_map_order(np.where(qualified,0,-1))
        validated=TensorMap(maps=maps,phases={0:result['uc']},steps=[1,result['step'],result['step']])
        # Preserve tomography labels when available, as in the standard visualisation notebook.
        try:
            tmap_tomo = TensorMap.from_h5(ds.grainsfile, h5group='TensorMap_' + phase_str)
            if tmap_tomo.labels.shape != validated.phase_ids.shape:
                raise ValueError('Tomography labels and refined TensorMap shapes differ')
            validated.add_map('labels', np.where(validated.phase_ids >= 0, tmap_tomo.labels, -1))
        except (FileNotFoundError, OSError, KeyError):
            print('No tomography labels available for', label)
        out=VALIDATION/(NAME+'_'+label+'_qualified_tmap.h5')
        if out.exists():
            existing=TensorMap.from_h5(str(out))
            np.testing.assert_allclose(existing.UBI,validated.UBI,equal_nan=True)
            np.testing.assert_array_equal(existing.phase_ids,validated.phase_ids)
        else:
            validated.to_h5(str(out)); validated.to_paraview(str(out)); validated.to_ctf_mtex(str(out).replace('.h5','.ctf'))
        validated.get_ipf_maps()
        result['qualified']=qualified; result['angle']=angular; result['strain_delta']=strain_delta; result['validated']=validated; result['output']=out
        SUMMARIES[label]=summary
        fig,axes=plt.subplots(1,3,figsize=(15,4.5),layout='constrained')
        ipf=TensorMap.map_order_to_recon_order(validated['ipf_z'],0)
        axes[0].imshow(np.where(qualified[...,None],np.nan_to_num(ipf),1),origin='lower'); axes[0].set_title(label+' qualified IPF-Z')
        im=axes[1].imshow(angular,origin='lower',vmin=0,vmax=0.5,cmap='magma'); fig.colorbar(im,ax=axes[1],label='split orientation difference (deg)'); axes[1].set_title('No smoothing; colour clipped at 0.5 deg')
        strain_vmax = np.nanpercentile(strain_delta,99) if strain_common.any() else None
        im=axes[2].imshow(strain_delta,origin='lower',vmin=0,vmax=strain_vmax,cmap='viridis'); fig.colorbar(im,ax=axes[2],label='Frobenius strain difference'); axes[2].set_title('Strain repeatability, not absolute accuracy')
        snap(fig,label+'_physical_validation')
print('Qualified maps and diagnostics:',VALIDATION)

# Orientation, Support and Strain

Standard S3DXRD views below use the split-qualified TensorMap. Stored strains are the directional-strain fits from refinement, relative to its calibration reference cell; changing TensorMap phases does not re-reference them. They are dimensionless internally and displayed in microstrain (1e-6). Off-diagonal entries are tensor shear, not engineering shear. Unit-cell summaries are derived separately from UBI. No stress is inferred. Colour limits affect display only; unsupported voxels remain masked. Split agreement is repeatability, not absolute strain accuracy.

In [ ]:
from pprint import pprint
for label, result in RESULTS.items():
    tmap = result['validated']
    for field in ('nuniq', 'ipf_x', 'ipf_y', 'ipf_z'):
        tmap.plot(field)
    eul = tmap.euler
    supported_cells = tmap.unitcell[tmap.phase_ids >= 0]
    print(label, 'reference cell:')
    pprint(result['uc'])
    if supported_cells.size:
        print('Mean split-qualified refined cell:')
        pprint(dict(zip(('a (Angstrom)', 'b (Angstrom)', 'c (Angstrom)', 'alpha (deg)', 'beta (deg)', 'gamma (deg)'), np.nanmean(supported_cells, axis=0).tolist())))
    else:
        print('No split-qualified voxels for unit-cell statistics')


In [ ]:
# Display only: robust automatic limits; saturated tails remain in the data and metrics.
strain_limit_microstrain = None
strain_display_percentile = 99.0  # use 100 for the full range
strain_components = ((0, 0), (1, 1), (2, 2), (0, 1), (0, 2), (1, 2))
for label, result in RESULTS.items():
    eps = TensorMap.map_order_to_recon_order(result['validated'].eps_sample, 0)
    valid = result['qualified'] & np.isfinite(eps).all(axis=(-2, -1))
    if not valid.any():
        print(label, 'no split-qualified finite strain; skipping strain plots')
        continue
    eps = np.where(valid[..., None, None], eps, np.nan)
    limit = strain_limit_microstrain
    if limit is None:
        component_values = np.stack([eps[..., i, j][valid] for i, j in strain_components])
        limit = float(np.percentile(np.abs(component_values), strain_display_percentile)) * 1e6
    if not np.isfinite(limit) or limit <= 0:
        limit = 1.0  # display range for identically zero fields only
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True, layout='constrained')
    for ax, (i, j) in zip(axes.flat, strain_components):
        im = ax.imshow(eps[..., i, j] * 1e6, origin='lower', cmap='RdBu_r', vmin=-limit, vmax=limit, interpolation='nearest')
        ax.set_title('eps_%s%s' % ('xyz'[i], 'xyz'[j]))
    fig.colorbar(im, ax=axes.ravel().tolist(), label='Sample-frame tensor strain (microstrain)', extend='both')
    print(label, 'component display limits (microstrain):', -limit, limit, '; saturated tails retained in metrics')
    fig.supxlabel('Reconstruction column index'); fig.supylabel('Reconstruction row index')
    snap(fig, label + '_strain_components')
    hydrostatic = TensorMap.map_order_to_recon_order(result['validated'].eps_hydro, 0)[..., 0, 0]
    deviatoric = TensorMap.map_order_to_recon_order(result['validated'].eps_devia, 0)
    hydrostatic = np.where(valid, hydrostatic, np.nan)
    deviatoric = np.where(valid[..., None, None], deviatoric, np.nan)
    equivalent = np.sqrt((2.0 / 3.0) * np.sum(deviatoric ** 2, axis=(-2, -1)))
    hydro_limit = strain_limit_microstrain
    equivalent_limit = strain_limit_microstrain
    if strain_limit_microstrain is None:
        hydro_limit = float(np.percentile(np.abs(hydrostatic[valid]), strain_display_percentile)) * 1e6
        equivalent_limit = float(np.percentile(equivalent[valid], strain_display_percentile)) * 1e6
    hydro_limit = hydro_limit if np.isfinite(hydro_limit) and hydro_limit > 0 else 1.0
    equivalent_limit = equivalent_limit if np.isfinite(equivalent_limit) and equivalent_limit > 0 else 1.0
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout='constrained')
    im = axes[0].imshow(hydrostatic * 1e6, origin='lower', cmap='RdBu_r', vmin=-hydro_limit, vmax=hydro_limit, interpolation='nearest')
    axes[0].set_title('Mean normal strain: trace(eps)/3'); fig.colorbar(im, ax=axes[0], label='microstrain', extend='both')
    im = axes[1].imshow(equivalent * 1e6, origin='lower', cmap='viridis', vmin=0, vmax=equivalent_limit, interpolation='nearest')
    axes[1].set_title('Equivalent deviatoric strain'); fig.colorbar(im, ax=axes[1], label='microstrain', extend='max')
    for i, j in strain_components:
        axes[2].hist(eps[..., i, j][valid] * 1e6, bins=60, range=(-limit, limit), histtype='step', label='eps_%s%s' % ('xyz'[i], 'xyz'[j]))
    axes[2].set(xlabel='Tensor strain (microstrain; display range only)', ylabel='Qualified voxel count'); axes[2].legend()
    snap(fig, label + '_strain_invariants_histograms')


# Split Strain Precision Diagnostics

Compare sample-frame tensor components using delta = split0 - split1, in microstrain (1e-6); shear is tensor shear, not engineering shear. Report the existing common finite non-placeholder full/split strain support **before** angular qualification and the existing qualified support (split angle <= 0.5 degrees). These are voxel-weighted, conditional repeatability diagnostics; comparing both supports exposes selection bias, without changing acceptance. Voxels are not independent replicates, and no standard error of the mean is inferred.

Bias is mean(delta); sigma_delta is the centred sample standard deviation (ddof=1). Robust scatter is 1.482602218505602 * median(abs(delta - median(delta))), the Gaussian-consistent MAD factor 1/Phi^-1(0.75), not a finite-sample correction. RMS = sqrt(mean(delta**2)) includes bias; absolute-difference percentiles are p50, p95 and p99. Counts show requested support and finite pairs per component. Empty statistics are NaN; for one pair sigma_delta and its derived estimates are NaN (MAD is zero).

**Conditional estimates only:** under independent, equal-variance, balanced split noise, per-half precision is sigma_delta/sqrt(2) and full-data precision is sigma_delta/2, assuming the full-data estimator behaves like the equally weighted mean of the two halves. The modulo-group split does not establish these assumptions or balanced information. These estimates are not certified uncertainty, absolute accuracy, strain detection limits or spatial resolution. Shared diffraction origins, calibration, reference cell and selection can induce correlation or common-mode errors: Var(delta) = Var(split0) + Var(split1) - 2 Cov(split0, split1). Split agreement cannot detect shared bias.

Metrics are displayed and retained in STRAIN_SPLIT_METRICS only; existing HDF5 diagnostics and map export contracts are unchanged.

In [ ]:
def split_strain_metrics(eps0, eps1, support, components):
    """Component delta statistics in microstrain; conditional sigmas need the stated noise assumptions."""
    import numpy as np
    support = np.asarray(support, dtype=bool)
    eps0, eps1 = np.asarray(eps0, dtype=float), np.asarray(eps1, dtype=float)
    rows = {}
    for i, j in components:
        x, y = eps0[..., i, j][support], eps1[..., i, j][support]
        with np.errstate(invalid='ignore', over='ignore'):
            delta = (x - y) * 1e6
        delta = delta[np.isfinite(x) & np.isfinite(y) & np.isfinite(delta)]
        n = int(delta.size)
        sigma = float(np.std(delta, ddof=1)) if n > 1 else np.nan
        percentiles = np.percentile(np.abs(delta), [50, 95, 99]) if n else np.full(3, np.nan)
        rows['eps_%s%s' % ('xyz'[i], 'xyz'[j])] = dict(
            n_support=int(support.sum()), n_finite=n,
            bias=float(np.mean(delta)) if n else np.nan, sigma_delta=sigma,
            mad_sigma_delta=float(1.482602218505602 * np.median(np.abs(delta - np.median(delta)))) if n else np.nan,
            rms_delta=float(np.sqrt(np.mean(delta ** 2))) if n else np.nan,
            abs_p50=float(percentiles[0]), abs_p95=float(percentiles[1]), abs_p99=float(percentiles[2]),
            conditional_sigma_half=sigma / np.sqrt(2.0), conditional_sigma_full=sigma / 2.0)
    return rows

def display_split_strain_summary(label, support_name, rows, n_support, n_total):
    from IPython.display import display, Markdown
    heading = 'Common Support Before Angular Qualification' if support_name == 'common_before_angle' else 'Qualified Support (split angle <= 0.5 deg)'
    lines = ['### %s: %s' % (label, heading), '', 'Voxels: %d of %d. Delta = split0 - split1; tensor shear, not engineering shear.' % (n_support, n_total), '']
    groups = (
        ('Delta statistics', (('n_support', 'Support (voxels)'), ('n_finite', 'Finite (pairs)'), ('bias', 'Bias (microstrain)'), ('sigma_delta', 'sigma_delta (microstrain)'), ('mad_sigma_delta', 'MAD scatter (microstrain)'), ('rms_delta', 'RMS (microstrain)'))),
        ('Absolute delta percentiles', (('abs_p50', 'p50 (microstrain)'), ('abs_p95', 'p95 (microstrain)'), ('abs_p99', 'p99 (microstrain)'))),
        ('Conditional precision (assumptions above; not certified uncertainty or resolution)', (('conditional_sigma_half', 'Per-half sigma (microstrain)'), ('conditional_sigma_full', 'Full-data sigma (microstrain)'))))
    for title, columns in groups:
        lines += ['#### ' + title, '', '| Component | ' + ' | '.join(header for key, header in columns) + ' |', '| :--- | ' + ' | '.join('---:' for key, header in columns) + ' |']
        for component, row in rows.items():
            values = [('%d' if key in ('n_support', 'n_finite') else '%.3g') % row[key] for key, header in columns]
            lines.append('| ' + component + ' | ' + ' | '.join(values) + ' |')
        lines.append('')
    display(Markdown('\n'.join(lines)))


In [ ]:
STRAIN_SPLIT_METRICS = {}
split_strain_limit_microstrain = None  # display only; None: automatic per component
split_strain_display_percentile = 99.0  # absolute finite pre-angle delta; 100: full range
print('Delta = split0 - split1; all statistics below are microstrain, except voxel counts.')
print('sigma_delta: sample std (ddof=1); MAD Gaussian factor: 1.482602218505602.')
print('Conditional sigmas ONLY under independent equal-variance balanced split noise and a full-data mean estimator; not certified uncertainty or resolution.')
print('Shared origins/calibration induce correlation; compare pre-angle and qualified support for selection bias.')
print('Histogram display only: tails outside the plotted range are omitted, not clipped; all metric values unchanged.')
for label, result in RESULTS.items():
    a, b = result['splits']
    common_strain = result['valid'] & a['valid'] & b['valid'] & result['strain_ok'] & a['strain_ok'] & b['strain_ok']
    for eps_field in (result['eps'], a['eps'], b['eps']):
        common_strain = common_strain & np.isfinite(eps_field).all(axis=(-2, -1))
    supports = (('common_before_angle', common_strain), ('qualified', result['qualified'] & common_strain))
    STRAIN_SPLIT_METRICS[label] = {}
    for support_name, support in supports:
        rows = split_strain_metrics(a['eps'], b['eps'], support, strain_components)
        STRAIN_SPLIT_METRICS[label][support_name] = rows
        display_split_strain_summary(label, support_name, rows, int(support.sum()), support.size)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), layout='constrained')
    for ax, (i, j) in zip(axes.flat, strain_components):
        component = 'eps_%s%s' % ('xyz'[i], 'xyz'[j])
        with np.errstate(invalid='ignore', over='ignore'):
            delta = (a['eps'][..., i, j][common_strain] - b['eps'][..., i, j][common_strain]) * 1e6
        finite = np.isfinite(delta)
        limit = split_strain_limit_microstrain
        if limit is None:
            limit = float(np.percentile(np.abs(delta[finite]), split_strain_display_percentile)) if finite.any() else 1.0
        if not np.isfinite(limit) or limit <= 0:
            limit = 1.0
        bins = np.linspace(-limit, limit, 61)
        for support_name, support in supports:
            values = delta[finite & support[common_strain]]
            outside = int(np.count_nonzero(np.abs(values) > limit))
            print(label, component, support_name, 'outside display range:', outside, 'of', values.size, '; limits (microstrain):', -limit, limit)
            if values.size:
                ax.hist(values, bins=bins, histtype='step', label=support_name + ' (n=%d)' % values.size)
        ax.axvline(0, color='grey', linewidth=0.8)
        ax.set(title=component, xlabel='split0 - split1 (microstrain; display range only)', ylabel='Voxel count', xlim=(-limit, limit))
        if finite.any():
            ax.legend(fontsize='small')
        else:
            ax.text(0.5, 0.5, 'No common finite strain pairs', transform=ax.transAxes, ha='center')
    fig.suptitle(label + ': split component differences, before/after angular selection')
    snap(fig, label + '_split_strain_component_differences')


# Spatial Sampling, Not Measured Resolution

Report DataSet binning metadata and the effective beam parameter from the PBPRefine object loaded above through ImageD11 APIs. The nominal dty/omega bin widths are derived by DataSet.guessbins; they do not establish regular acquisition spacing (scans can be irregular/interlaced and bins can include padding). Native dty motor units are retained, not assumed to be mm or micrometres. The reconstruction XY grid pitch is sampling, not a measured resolving power; the singleton Z step is ordinal, not a measured Z spacing.

PBPRefine.beam_size is an **assumed** beam width used for distance weighting, with regularisation length beam_size/3; the API defaults to ystep when no beam size was saved. It is not a PSF measurement. Actual spatial resolution **has not been measured** here: no PSF or edge-response benchmark is available. Neither smoothness, split agreement, scan step nor assumed beam width certifies spatial resolution.

In [ ]:
from pprint import pprint
for label, result in RESULTS.items():
    sampling = result['spatial_sampling']
    print(label, 'spatial sampling metadata (native dty motor units; omega in degrees)')
    pprint({
        'DataSet nominal dty bin step (native dty motor units; derived, not measured resolution)': '%.3g' % sampling['dty_step'],
        'DataSet nominal omega bin width (deg; may be irregular/interlaced)': '%.3g' % sampling['omega_bin_width_deg'] if sampling['omega_bin_width_deg'] is not None else None,
        'Reconstruction XY grid pitch (native dty motor units; Z step is ordinal only)': '%.3g' % result['step'],
        'Effective assumed PBPRefine beam_size (native dty motor units; saved or API default ystep)': '%.3g' % sampling['assumed_beam_size'],
        'Distance-weight regularisation length beam_size/3 (native dty motor units)': '%.3g' % (sampling['assumed_beam_size'] / 3.0),
    }, width=100)
    print('  Actual spatial resolution not measured: no PSF/edge benchmark; sampling and assumed beam width are not resolution.')


# Export
## TensorMap

Export the tomography-supported orientation-only TensorMap and completion hypotheses in physical XY units, with matching Paraview XDMF files.

In [ ]:
# Separate repeatability-qualified completion from tomography-supported orientation.
r=RESULTS['conservative']
ds=dataset.load(str(VARIANT_DATASET))
tomo=TensorMap.from_h5(ds.grainsfile,h5group='TensorMap_'+PHASE)
tomo_support=TensorMap.map_order_to_recon_order(tomo.phase_ids,0)>=0
primary=r['qualified']&tomo_support
completion=r['qualified']&~tomo_support
exported_maps={}
for kind,support in [('orientation_supported',primary),('completion_hypotheses',completion)]:
    source=r['validated']
    maps={key:np.array(source[key],copy=True) for key in ('U','phase_ids','nuniq')}
    invalid=~TensorMap.recon_order_to_map_order(support)
    maps['U'][invalid]=np.nan; maps['phase_ids'][invalid]=-1; maps['nuniq'][invalid]=0
    out=TensorMap(maps=maps,phases=source.phases,steps=[1,ds.ystep*1000,ds.ystep*1000])
    out.get_ipf_maps()
    path=VALIDATION/(NAME+'_'+kind+'_um_tmap.h5')
    assert not path.exists(),path
    out.to_h5(str(path)); out.to_paraview(str(path))
    exported_maps[kind]=(out,path)
    with h5py.File(path,'r+') as h:
        p=h.create_group('workflow_provenance')
        p.attrs['seed_policy']='fresh Anri-only global and local hypotheses'
        p.attrs['source_diagnostics']=str(DIAG)
        p.attrs['spatial_xy_units']='um'
        p.attrs['z_step_is_ordinal_placeholder']=True
        p.attrs['conditional_not_blind_validation']=True
        p.attrs['strain_or_measured_local_lattice_not_included']=True
        p.attrs['ctf_band_mad_columns_are_placeholders']=True
        p.attrs['supported_voxels']=int(support.sum())
    loaded=TensorMap.from_h5(str(path))
    np.testing.assert_array_equal(TensorMap.map_order_to_recon_order(loaded.phase_ids,0)>=0,support)
    print(kind,int(support.sum()),path)
    out.plot('ipf_z')


# MTEX

Export the supported orientation map and completion hypotheses to CTF. Band/MAD columns are placeholders; use HDF5 measured-event counts and split diagnostics for support assessment.

In [ ]:
for kind,(tmap,path) in exported_maps.items():
    ctf_path=path.with_suffix('.ctf')
    tmap.to_ctf_mtex(str(ctf_path))
    print(kind,ctf_path)

In [ ]:
# Check that experimental inputs and the manual mask are unchanged.
import json
manifest=json.loads((RUN/'input_manifest.json').read_text())
for filename,expected in manifest['inputs'].items():
    digest=hashlib.sha256()
    with Path(filename).open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''): digest.update(block)
    assert digest.hexdigest()==expected,filename
with h5py.File(manifest.get('manual_mask_source_file',str(SOURCE_GRAINS))) as h: manual=h['masks/'+phase_str][()]
assert hashlib.sha256(np.ascontiguousarray(manual).tobytes()).hexdigest()==manifest['manual_mask_sha256']
assert not np.any(primary&~manual)
print('Experimental inputs and manual mask unchanged')
# Visualisation does not change scientific DataSet values; do not rewrite hashed inputs.
